# Online Retail Intelligence Capstone
## Notebook 02: Data Cleaning and ETL

### Objective
Prepare a reliable dataset for sales performance analysis,
statistical testing, and customer segmentation.

### Cleaning Strategy
1. Load the original Kaggle dataset.
2. Preserve the raw records.
3. Standardise data types.
4. Investigate and handle duplicate records.
5. Separate valid sales from cancellations and returns.
6. Handle missing product descriptions.
7. Flag potentially unreliable customer identifiers.
8. Create revenue and date-related features.
9. Validate and export the cleaned datasets.

### Expected Outputs
- Cleaned sales transactions for business intelligence.
- Customer-level transactions for RFM segmentation.
- A documented summary of cleaning decisions.

In [47]:
# STEP 1: Import libraries for data manipulation and file management.
import pandas as pd
import numpy as np
import kagglehub

from pathlib import Path

# STEP 2: Define the Kaggle dataset identifier.
# This ensures the cleaning notebook uses the same data source.
dataset_id = "abhishekrp1517/online-retail-transactions-dataset"

# STEP 3: Retrieve the dataset from KaggleHub's local cache.
# If the dataset is not cached, KaggleHub downloads it.
dataset_path = Path(
    kagglehub.dataset_download(dataset_id)
)

# STEP 4: Identify the source CSV file.
# Verify that exactly one CSV file is available.
csv_files = list(dataset_path.rglob("*.csv"))

if len(csv_files) != 1:
    raise ValueError(
        f"Expected one CSV file, found {len(csv_files)}."
    )

# STEP 5: Load the raw dataset without modifying its contents.
df_raw = pd.read_csv(
    csv_files[0],
    encoding="ISO-8859-1"
)

# STEP 6: Create a working copy for cleaning.
# This preserves the original DataFrame for comparison.
df_clean = df_raw.copy()

# STEP 7: Confirm the number of records before cleaning.
print("Raw dataset shape:", df_raw.shape)
print("Working dataset shape:", df_clean.shape)

display(df_clean.head())

Raw dataset shape: (541909, 8)
Working dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


### Data Type Standardisation

Before cleaning transactions, the dataset's column types must
be standardised to support accurate calculations and filtering.

- InvoiceDate will be converted to datetime.
- InvoiceNo and StockCode will be treated as identifiers.
- CustomerID will be stored as a nullable integer identifier.
- Quantity and UnitPrice will remain numeric.

The original DataFrame will remain unchanged.

In [48]:
# STEP 8: Convert InvoiceDate into a datetime data type.
# This enables accurate date filtering and time-series analysis.
# Invalid dates are converted to NaT for subsequent validation.
df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"],
    errors="coerce"
)

# STEP 9: Standardise invoice and product identifiers.
# String types preserve identifier formats and allow text operations.
df_clean["InvoiceNo"] = (
    df_clean["InvoiceNo"].astype("string").str.strip()
)

df_clean["StockCode"] = (
    df_clean["StockCode"].astype("string").str.strip()
)

# STEP 10: Standardise customer identifiers.
# The nullable Int64 type supports missing IDs if present.
df_clean["CustomerID"] = (
    pd.to_numeric(df_clean["CustomerID"], errors="coerce")
    .astype("Int64")
)

# STEP 11: Verify the updated data types.
# Check for any invalid dates or customer IDs introduced by conversion.
print("UPDATED DATA TYPES")
print(df_clean.dtypes)

print("\nInvalid invoice dates:",
      df_clean["InvoiceDate"].isna().sum())

print("Missing customer IDs:",
      df_clean["CustomerID"].isna().sum())

# STEP 12: Confirm the original dataset remains unchanged.
# Only the working copy should have updated data types.
print("\nOriginal InvoiceDate type:",
      df_raw["InvoiceDate"].dtype)

print("Cleaned InvoiceDate type:",
      df_clean["InvoiceDate"].dtype)

print("\nWorking dataset shape:", df_clean.shape)

UPDATED DATA TYPES
InvoiceNo      string[python]
StockCode      string[python]
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              Int64
Country                object
dtype: object

Invalid invoice dates: 0
Missing customer IDs: 0

Original InvoiceDate type: object
Cleaned InvoiceDate type: datetime64[ns]

Working dataset shape: (541909, 8)


### Duplicate Transaction Investigation

Duplicate records can inflate sales revenue, product quantities,
and customer purchasing frequency.

This section identifies exact duplicate records and examines
their characteristics before deciding how to handle them.

Identical transaction lines may occasionally represent genuine
repeated purchases, so duplicate removal requires a documented
analytical decision.

In [49]:
# STEP 13: Identify exact duplicate records.
# duplicated() checks whether a row matches an earlier row
# across all columns.
duplicate_mask = df_clean.duplicated()

# STEP 14: Count duplicate records.
# This establishes how many rows would be removed if
# we retained only the first occurrence.
duplicate_count = duplicate_mask.sum()

print("Exact duplicate rows:", duplicate_count)

# STEP 15: Calculate the percentage of duplicate records.
# This measures the scale of the issue.
duplicate_percentage = (
    duplicate_count / len(df_clean)
) * 100

print(
    "Duplicate percentage:",
    round(duplicate_percentage, 2),
    "%"
)

# STEP 16: Display sample duplicate groups.
# keep=False includes both the original and repeated rows.
# Sorting by invoice number helps us compare matching records.
duplicate_records = df_clean[
    df_clean.duplicated(keep=False)
].sort_values("InvoiceNo")

print("\nSample duplicate records:")
display(duplicate_records.head(10))

# STEP 17: Check the number of unique invoices containing duplicates.
# This shows how widely duplicates are distributed across orders.
duplicate_invoices = duplicate_records["InvoiceNo"].nunique()

print(
    "\nInvoices containing duplicate records:",
    duplicate_invoices
)

# STEP 18: Confirm no records have been removed yet.
# We are investigating duplicates before applying cleaning rules.
print("\nCurrent dataset shape:", df_clean.shape)

Exact duplicate rows: 5268
Duplicate percentage: 0.97 %

Sample duplicate records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
604,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920,United Kingdom
605,536412,22902,TOTE BAG I LOVE LONDON,7,2010-12-01 11:49:00,2.10,17920,United Kingdom



Invoices containing duplicate records: 1933

Current dataset shape: (541909, 8)


### Duplicate Handling Decision

The dataset contains 5,268 exact duplicate rows (0.97%),
distributed across 1,933 invoices.

Repeated transaction lines may inflate revenue, quantities
sold, and customer purchasing metrics. However, identical
rows may occasionally represent legitimate purchases.

For the primary analytical dataset, exact duplicates will
be removed while retaining the first occurrence.

The original dataset will remain unchanged, and the number
of removed records will be documented.

**Limitation:** Without a unique transaction-line identifier,
it is not possible to confirm whether every repeated row
is erroneous.

In [50]:
# STEP 19: Record the number of rows before removing duplicates.
# This establishes a baseline for measuring the cleaning impact.
rows_before = len(df_clean)

# STEP 20: Remove exact duplicate records from the working dataset.
# keep="first" retains the first occurrence of each identical row.
# The original df_raw DataFrame remains unchanged.
df_clean = df_clean.drop_duplicates(
    keep="first"
).copy()

# STEP 21: Calculate the number of removed records.
# This provides an auditable measure of the cleaning operation.
rows_after = len(df_clean)
removed_duplicates = rows_before - rows_after

# STEP 22: Verify that no exact duplicate rows remain.
# This confirms that the cleaning operation worked correctly.
remaining_duplicates = df_clean.duplicated().sum()

# STEP 23: Display the cleaning results.
print("Rows before cleaning:", rows_before)
print("Duplicates removed:", removed_duplicates)
print("Rows after cleaning:", rows_after)
print("Remaining duplicates:", remaining_duplicates)

# STEP 24: Confirm that the original dataset is unchanged.
# Preserving the original data supports reproducibility.
print("\nOriginal dataset shape:", df_raw.shape)
print("Cleaned dataset shape:", df_clean.shape)

Rows before cleaning: 541909
Duplicates removed: 5268
Rows after cleaning: 536641
Remaining duplicates: 0

Original dataset shape: (541909, 8)
Cleaned dataset shape: (536641, 8)


### Transaction Classification

The dataset contains sales transactions, cancellations,
returns, and potentially non-sales adjustments.

To avoid overstating sales revenue, transactions will be
classified using invoice numbers, quantities, and unit prices.

Classification rules:

- Cancellation: InvoiceNo begins with "C".
- Return/negative adjustment: Non-cancellation invoice with Quantity <= 0.
- Non-positive price: Remaining transaction with UnitPrice <= 0.
- Valid sale: Non-cancellation transaction with positive
  Quantity and UnitPrice.

These categories are mutually exclusive. Cancellation
records are classified first to avoid double-counting.

The classified dataset will be retained so that excluded
records remain available for further investigation.

In [51]:
# STEP 25: Identify invoices marked as cancellations.
# Convert the result to a NumPy Boolean array for compatibility
# with np.select(). Missing values are treated as False.
is_cancellation = (
    df_clean["InvoiceNo"]
    .str.upper()
    .str.startswith("C", na=False)
    .to_numpy(dtype=bool, na_value=False)
)

# STEP 26: Identify records with non-positive quantities.
# These may represent returns or other inventory adjustments.
has_nonpositive_quantity = (
    (df_clean["Quantity"] <= 0)
    .fillna(False)
    .to_numpy(dtype=bool)
)

# STEP 27: Identify records with non-positive unit prices.
# Such records should not contribute to completed-sales revenue.
has_nonpositive_price = (
    (df_clean["UnitPrice"] <= 0)
    .fillna(False)
    .to_numpy(dtype=bool)
)

# STEP 28: Classify transactions into mutually exclusive categories.
# Conditions are evaluated in order, with cancellations taking priority.
conditions = [
    is_cancellation,
    has_nonpositive_quantity,
    has_nonpositive_price
]

categories = [
    "Cancellation",
    "Return/Negative Adjustment",
    "Non-Positive Price"
]

df_clean["TransactionType"] = np.select(
    conditions,
    categories,
    default="Valid Sale"
)

# STEP 29: Summarise the number of records in each category.
# This helps identify how many rows qualify for sales analysis.
transaction_summary = (
    df_clean["TransactionType"]
    .value_counts()
    .rename_axis("Transaction Type")
    .reset_index(name="Number of Rows")
)

print("TRANSACTION CLASSIFICATION SUMMARY")
display(transaction_summary)

# STEP 30: Verify that every transaction has been classified.
# The category counts must equal the total number of cleaned rows.
assert transaction_summary["Number of Rows"].sum() == len(df_clean)

print("\nTotal classified rows:", len(df_clean))
print("Classification validation: PASSED")

TRANSACTION CLASSIFICATION SUMMARY


,Transaction Type,Number of Rows
0,Valid Sale,524878
1,Cancellation,9251
2,Return/Negative Adjustment,1336
3,Non-Positive Price,1176



Total classified rows: 536641
Classification validation: PASSED


### Transaction Classification Findings

Following the removal of 5,268 exact duplicate rows,
536,641 transaction records remained.

The transaction classification identified:

- 524,878 valid-sale records (97.81%).
- 9,251 cancellation-related records (1.72%).
- 1,336 return or negative-adjustment records (0.25%).
- 1,176 non-positive-price records (0.22%).

The categories are mutually exclusive, with cancellations
receiving priority during classification.

**Analytical decision:** Valid-sale records will form the
basis of the primary sales performance dataset. Other
transaction categories will remain available in the
classified dataset for investigation and reporting.

These classifications are rule-based and do not independently
confirm the final payment status of transactions.

In [52]:
# STEP 31: Create a separate DataFrame containing valid sales.
# This excludes cancellations, negative adjustments, and
# non-positive prices from the primary sales analysis.
df_sales = df_clean.loc[
    df_clean["TransactionType"] == "Valid Sale"
].copy()

# STEP 32: Create a separate DataFrame for excluded transactions.
# Keeping these records supports transparency and further analysis.
df_excluded = df_clean.loc[
    df_clean["TransactionType"] != "Valid Sale"
].copy()

# STEP 33: Calculate revenue for each valid sales transaction line.
# Line revenue equals the quantity purchased multiplied by unit price.
df_sales["Revenue"] = (
    df_sales["Quantity"] * df_sales["UnitPrice"]
)

# STEP 34: Verify that the two datasets account for all classified rows.
# No transaction should be lost during separation.
assert len(df_sales) + len(df_excluded) == len(df_clean)

# STEP 35: Validate the revenue calculations.
# Valid sales must have positive quantities, prices, and revenue.
assert (df_sales["Quantity"] > 0).all()
assert (df_sales["UnitPrice"] > 0).all()
assert (df_sales["Revenue"] > 0).all()

# STEP 36: Display the results.
# This confirms the dataset sizes and initial sales revenue.
print("Valid sales rows:", len(df_sales))
print("Excluded transaction rows:", len(df_excluded))
print("Total classified rows:", len(df_clean))

print(
    "\nTotal valid-sales revenue:",
    round(df_sales["Revenue"].sum(), 2)
)

print("\nValidation: PASSED")

# Preview the valid-sales dataset with its new Revenue column.
display(df_sales.head())

Valid sales rows: 524878
Excluded transaction rows: 11763
Total classified rows: 536641

Total valid-sales revenue: 10642110.8

Validation: PASSED


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionType,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Valid Sale,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Valid Sale,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Valid Sale,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Valid Sale,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Valid Sale,20.34


### Valid Sales Dataset Findings

After removing exact duplicates and classifying transactions,
524,878 records met the defined valid-sales criteria.

A further 11,763 records were excluded from the primary
sales dataset because they were classified as cancellations,
negative adjustments, or non-positive-price transactions.

Line revenue was calculated as:

**Revenue = Quantity × UnitPrice**

The total valid-sales revenue was **10,642,110.80**.

All validation checks passed, confirming that:
- Valid and excluded records reconcile to the classified dataset.
- Valid sales have positive quantities and unit prices.
- Calculated sales-line revenues are positive.

**Limitation:** This revenue represents positive sales-line
value before accounting for returns, refunds, or other
financial adjustments.

In [53]:
# STEP 37: Identify missing or blank product descriptions.
# Both null values and descriptions containing only spaces
# are treated as missing product information.
missing_description_mask = (
    df_sales["Description"].isna()
    | df_sales["Description"].astype("string").str.strip().eq("")
        .fillna(False)
)

# STEP 38: Count records with missing descriptions.
# This measures how many valid sales require a replacement label.
missing_description_count = missing_description_mask.sum()

print(
    "Missing or blank descriptions before cleaning:",
    missing_description_count
)

# STEP 39: Record revenue before updating descriptions.
# Description cleaning should not change financial calculations.
revenue_before = df_sales["Revenue"].sum()
rows_before = len(df_sales)

# STEP 40: Standardise available descriptions.
# Strip unnecessary spaces while preserving product wording.
df_sales["Description"] = (
    df_sales["Description"]
    .astype("string")
    .str.strip()
)

# STEP 41: Replace missing or blank descriptions.
# StockCode remains available for identifying affected products.
df_sales["Description"] = (
    df_sales["Description"]
    .fillna("Unknown Product")
    .replace("", "Unknown Product")
)

# STEP 42: Verify that missing descriptions have been handled.
remaining_missing = (
    df_sales["Description"].isna().sum()
    + df_sales["Description"].eq("").sum()
)

# STEP 43: Confirm that financial data and row counts are unchanged.
assert len(df_sales) == rows_before
assert np.isclose(df_sales["Revenue"].sum(), revenue_before)
assert remaining_missing == 0

# STEP 44: Display the cleaning results.
print("\nDescriptions updated:", missing_description_count)
print("Remaining missing descriptions:", remaining_missing)
print("Sales rows after cleaning:", len(df_sales))
print("Revenue after cleaning:", round(df_sales["Revenue"].sum(), 2))

print("\nDescription cleaning validation: PASSED")

Missing or blank descriptions before cleaning: 0

Descriptions updated: 0
Remaining missing descriptions: 0
Sales rows after cleaning: 524878
Revenue after cleaning: 10642110.8

Description cleaning validation: PASSED


### Date Feature Engineering

Date-related features are created from InvoiceDate to support
time-series analysis and business intelligence reporting.

The following features will be generated:

- InvoiceYear: Year of the transaction.
- InvoiceMonth: Month number.
- YearMonth: Year and month in YYYY-MM format.
- InvoiceDay: Day of the month.
- DayOfWeek: Name of the weekday.
- InvoiceHour: Hour of the transaction.

These features will help identify sales trends, seasonality
and customer purchasing patterns.

The transformation must not change transaction counts or revenue.

In [54]:
# STEP 45: Extract the transaction year.
# This supports annual sales comparisons.
df_sales["InvoiceYear"] = df_sales["InvoiceDate"].dt.year

# STEP 46: Extract the transaction month.
# Month numbers allow chronological grouping and sorting.
df_sales["InvoiceMonth"] = df_sales["InvoiceDate"].dt.month

# STEP 47: Create a year-month identifier.
# YYYY-MM supports monthly sales analysis across different years.
df_sales["YearMonth"] = (
    df_sales["InvoiceDate"].dt.strftime("%Y-%m")
)

# STEP 48: Extract the day of the month.
# This helps investigate daily purchasing patterns.
df_sales["InvoiceDay"] = df_sales["InvoiceDate"].dt.day

# STEP 49: Extract the weekday name.
# This supports comparisons between different shopping days.
df_sales["DayOfWeek"] = (
    df_sales["InvoiceDate"].dt.day_name()
)

# STEP 50: Extract the transaction hour.
# This helps identify busy purchasing periods.
df_sales["InvoiceHour"] = df_sales["InvoiceDate"].dt.hour

# STEP 51: Validate the newly created date features.
# All valid-sales records should have complete date information.
date_columns = [
    "InvoiceYear",
    "InvoiceMonth",
    "YearMonth",
    "InvoiceDay",
    "DayOfWeek",
    "InvoiceHour"
]

assert df_sales[date_columns].notna().all().all()

# STEP 52: Verify that the original sales values are unchanged.
# Feature engineering should not alter transaction counts or revenue.
assert len(df_sales) == 524878
assert np.isclose(df_sales["Revenue"].sum(), revenue_before)

# STEP 53: Display the date features and validation results.
print("DATE FEATURE ENGINEERING")

display(
    df_sales[
        ["InvoiceDate"] + date_columns
    ].head()
)

print("\nSales records:", len(df_sales))
print("Total revenue:", round(df_sales["Revenue"].sum(), 2))
print("Date feature validation: PASSED")

DATE FEATURE ENGINEERING


,InvoiceDate,InvoiceYear,InvoiceMonth,YearMonth,InvoiceDay,DayOfWeek,InvoiceHour
0,2010-12-01 08:26:00,2010,12,2010-12,1,Wednesday,8
1,2010-12-01 08:26:00,2010,12,2010-12,1,Wednesday,8
2,2010-12-01 08:26:00,2010,12,2010-12,1,Wednesday,8
3,2010-12-01 08:26:00,2010,12,2010-12,1,Wednesday,8
4,2010-12-01 08:26:00,2010,12,2010-12,1,Wednesday,8



Sales records: 524878
Total revenue: 10642110.8
Date feature validation: PASSED


### Customer Identifier Quality Flag

The initial data quality assessment identified CustomerID
15287 as an unusually frequent identifier.

The identifier appeared in 135,101 original transaction
rows (24.93%) and was associated with invoices across
nine countries.

This raises concerns that the identifier may represent
a shared or substituted customer reference.

To protect customer-level analysis:

1. Preserve all eligible sales transactions.
2. Flag CustomerID 15287 for further investigation.
3. Create a provisional customer-analysis dataset excluding
   this identifier.
4. Retain the complete valid-sales dataset for revenue analysis.
5. Document the limitation and assess its impact during RFM
   segmentation.

The flag does not establish that the identifier is invalid.

In [55]:
# STEP 54: Identify the customer ID requiring investigation.
# This ID showed unusually high transaction frequency
# and appeared across multiple countries.
flagged_customer_id = 15287

# STEP 55: Flag transactions associated with this identifier.
# The Boolean flag preserves the original customer information.
df_sales["CustomerID_Flagged"] = (
    df_sales["CustomerID"] == flagged_customer_id
).fillna(False)

# STEP 56: Count flagged sales transaction rows.
# This measures the identifier's impact after sales cleaning.
flagged_rows = df_sales["CustomerID_Flagged"].sum()

flagged_percentage = (
    flagged_rows / len(df_sales)
) * 100

print("Flagged customer ID:", flagged_customer_id)
print("Flagged sales rows:", flagged_rows)
print(
    "Percentage of valid-sales rows:",
    round(flagged_percentage, 2),
    "%"
)

# STEP 57: Create a provisional dataset for customer analysis.
# Exclude the flagged identifier pending further validation.
# This dataset will be used for preliminary RFM analysis.
df_customer_sales = df_sales.loc[
    ~df_sales["CustomerID_Flagged"]
].copy()

# STEP 58: Compare the sales and customer-analysis datasets.
# The primary sales dataset retains all eligible transactions.
print("\nComplete valid-sales rows:", len(df_sales))
print("Provisional customer-analysis rows:", len(df_customer_sales))

# STEP 59: Verify that the datasets reconcile correctly.
# The difference must equal the number of flagged records.
assert (
    len(df_sales) - len(df_customer_sales)
) == flagged_rows

# STEP 60: Confirm the full sales dataset remains unchanged.
# Flagging customers must not change the sales revenue.
assert len(df_sales) == 524878
assert np.isclose(
    df_sales["Revenue"].sum(),
    10642110.80
)

print("\nCustomer identifier validation: PASSED")

Flagged customer ID: 15287
Flagged sales rows: 132206
Percentage of valid-sales rows: 25.19 %

Complete valid-sales rows: 524878
Provisional customer-analysis rows: 392672

Customer identifier validation: PASSED


### Customer Identifier Investigation Findings

CustomerID 15287 appears in 132,206 valid-sales records,
representing 25.19% of the valid-sales dataset.

Because this identifier has unusually high transaction
frequency and was previously observed across multiple
countries, it has been flagged for further investigation.

Two datasets have been retained:

- Complete sales dataset: 524,878 records.
- Provisional customer-analysis dataset: 392,672 records.

**Analytical decision:** The complete sales dataset will
support revenue and product performance analysis. The
provisional customer-analysis dataset excludes the flagged
identifier to reduce potential distortion in customer
segmentation.

**Limitation:** There is insufficient evidence to confirm
that CustomerID 15287 is invalid. Customer segmentation
results should therefore be compared with and without
this exclusion before final conclusions are drawn.

### Exporting Cleaned Data

The cleaned datasets will be exported to CSV format
for reproducibility and reuse.

Three outputs will be created:

1. Complete valid-sales dataset for business intelligence.
2. Provisional customer-analysis dataset for RFM segmentation.
3. Excluded transaction dataset for data-quality auditing.

The exports will be validated by reloading the CSV files
and comparing their row counts with the source DataFrames.

The original Kaggle dataset remains unchanged.

In [56]:
# STEP 61: Identify the project root directory.
# The notebook is located inside the jupyter_notebooks folder.
project_root = Path.cwd().resolve()

if project_root.name == "jupyter_notebooks":
    project_root = project_root.parent

# STEP 62: Create a folder for processed datasets.
# exist_ok=True prevents errors if the folder already exists.
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# STEP 63: Define descriptive filenames for each dataset.
# These names will be reused in subsequent analysis notebooks.
sales_path = processed_dir / "retail_sales_clean.csv"
customer_path = processed_dir / "retail_customer_sales.csv"
excluded_path = processed_dir / "retail_excluded_transactions.csv"

# STEP 64: Export the cleaned datasets to CSV files.
# index=False avoids saving Pandas row indexes as extra columns.
df_sales.to_csv(sales_path, index=False)
df_customer_sales.to_csv(customer_path, index=False)
df_excluded.to_csv(excluded_path, index=False)

# STEP 65: Reload the exported files.
# This checks that the files can be read successfully.
sales_check = pd.read_csv(sales_path)
customer_check = pd.read_csv(customer_path)
excluded_check = pd.read_csv(excluded_path)

# STEP 66: Validate the exported row counts.
# Every exported file must match its source DataFrame.
assert len(sales_check) == len(df_sales)
assert len(customer_check) == len(df_customer_sales)
assert len(excluded_check) == len(df_excluded)

# STEP 67: Validate the exported sales revenue.
# Reloading must not materially change the revenue total.
assert np.isclose(
    sales_check["Revenue"].sum(),
    df_sales["Revenue"].sum()
)

# STEP 68: Display export locations and validation results.
print("DATA EXPORT SUMMARY")
print("Sales dataset:", sales_path)
print("Customer dataset:", customer_path)
print("Excluded transactions:", excluded_path)

print("\nExported sales rows:", len(sales_check))
print("Exported customer rows:", len(customer_check))
print("Exported excluded rows:", len(excluded_check))

print("\nSales revenue:", round(sales_check["Revenue"].sum(), 2))
print("CSV export validation: PASSED")

C:\Users\Joy\AppData\Local\Temp\ipykernel_51560\109593168.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_check = pd.read_csv(sales_path)


DATA EXPORT SUMMARY
Sales dataset: C:\Users\Joy\OneDrive\Documents\vscode - projects\Online-Retail-Intelligence-Capstone\data\processed\retail_sales_clean.csv
Customer dataset: C:\Users\Joy\OneDrive\Documents\vscode - projects\Online-Retail-Intelligence-Capstone\data\processed\retail_customer_sales.csv
Excluded transactions: C:\Users\Joy\OneDrive\Documents\vscode - projects\Online-Retail-Intelligence-Capstone\data\processed\retail_excluded_transactions.csv

Exported sales rows: 524878
Exported customer rows: 392672
Exported excluded rows: 11763

Sales revenue: 10642110.8
CSV export validation: PASSED
